# BERT 트랙 전처리

**목표**: 3개 한국어 BERT 모델에 대해 토크나이징 + PyTorch Dataset/DataLoader 구성

| 모델 | HuggingFace ID |
|---|---|
| KoBERT | `skt/kobert-base-v1` |
| KLUE-RoBERTa | `klue/roberta-base` |
| KoELECTRA | `monologg/koelectra-base-v3-discriminator` |

**입력 형식**: `[CLS] 제목 [SEP] 본문 [SEP]`
- `truncation='only_second'` → 제목은 절대 자르지 않음, 본문만 자름
- `max_length=512`

**규칙**:
- `title_clean`, `content_clean` 그대로 사용 (추가 정제 없음)
- `test_final.parquet`은 봉인 — 절대 로딩하지 않음

## Step 0. 패키지 설치

BERT 토크나이저(`transformers`)와 PyTorch(`torch`)가 필요합니다.
- `sentencepiece`: KoBERT 토크나이저가 SentencePiece 기반이므로 필수
- `kobert-tokenizer`는 Python 3.14 미지원 → `AutoTokenizer` + `sentencepiece`로 대체

In [8]:
!pip install transformers torch sentencepiece protobuf

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1. 데이터 로딩

work_pool 3개 parquet을 병합합니다. (`test_final.parquet`은 봉인 — 로딩하지 않음)
- `work_pool_clickbait_auto.parquet` (106,014건)
- `work_pool_clickbait_direct.parquet` (40,106건)
- `work_pool_nonclickbait_auto.parquet` (145,346건)

BERT 트랙에서 사용할 컬럼: `title_clean`, `content_clean`, `binary_label`, `type_label`

In [ ]:
import pandas as pd

df = pd.concat([
    pd.read_parquet('data/processed/work_pool_clickbait_auto.parquet'),
    pd.read_parquet('data/processed/work_pool_clickbait_direct.parquet'),
    pd.read_parquet('data/processed/work_pool_nonclickbait_auto.parquet'),
], ignore_index=True)

print(f"전체 데이터: {len(df):,}건")
print(f"컬럼: {list(df.columns)}")
print(f"\n결측값 확인:")
print(df[['title_clean', 'content_clean', 'binary_label', 'type_label']].isnull().sum())
print(f"\n빈 문자열 확인:")
print(f"  title_clean 빈 문자열: {(df['title_clean'] == '').sum()}")
print(f"  content_clean 빈 문자열: {(df['content_clean'] == '').sum()}")
df.head(3)

전체 데이터: 291,466건
컬럼: ['newsID', 'newTitle', 'newsContent', 'binary_label', 'type_label', 'source_class', 'title_clean', 'content_clean']

결측값 확인:
title_clean      0
content_clean    0
binary_label     0
type_label       0
dtype: int64

빈 문자열 확인:
  title_clean 빈 문자열: 0
  content_clean 빈 문자열: 0


,newsID,newTitle,newsContent,binary_label,type_label,source_class,title_clean,content_clean
0,LC_M03_089986,RM이 부산서 본 이 전시 ...볼탕스키 작품이 말하는 것들,흉터는 의사들이 그다지 달가워하지 않는 시술이다.\n특별한 치료법이 없어 환자의 만...,1,-1,clickbait_auto,RM이 부산서 본 이 전시 ...볼탕스키 작품이 말하는 것들,흉터는 의사들이 그다지 달가워하지 않는 시술이다. 특별한 치료법이 없어 환자의 만족...
1,GB_M11_057148,"오미크론 250명 입원에도…英 당국 \""규제 계획 없어\""",미국 질병통제예방센터(CDC)가 모더나와 얀센 신종 코로나바이러스 감염증(코로나19...,1,-1,clickbait_auto,"오미크론 250명 입원에도…英 당국 \""규제 계획 없어\""",미국 질병통제예방센터(CDC)가 모더나와 얀센 신종 코로나바이러스 감염증(코로나19...
2,PO_M08_105090,"尹 '원전 안전성' 인터뷰 논란에 캠프 \""의미 다르게 전달됐다\""",북한이 12일 동해상으로 탄도미사일을 발사하며 윤석열정부 출범 이틀 만에 첫 무력도...,1,-1,clickbait_auto,"尹 '원전 안전성' 인터뷰 논란에 캠프 \""의미 다르게 전달됐다\""",북한이 12일 동해상으로 탄도미사일을 발사하며 윤석열정부 출범 이틀 만에 첫 무력도...


## Step 2. 라벨 분포 확인

- **이진 분류** (`binary_label`): 0=정상, 1=낚시성 → 약 50:50 균형 → `class_weight` 불필요
- **다중 분류** (`type_label`): 0~5 (Clickbait_Direct만), -1 (나머지) → 7.1배 불균형 → `class_weight='balanced'` 필수

In [ ]:
print("=" * 50)
print("이진 분류 (binary_label) 분포")
print("=" * 50)
binary_dist = df['binary_label'].value_counts().sort_index()
for label, count in binary_dist.items():
    name = "정상" if label == 0 else "낚시성"
    print(f"  {label} ({name}): {count:,}건 ({count/len(df)*100:.1f}%)")

print(f"\n{'=' * 50}")
print("다중 분류 (type_label) 분포 — Clickbait_Direct만")
print("=" * 50)
TYPE_NAMES = {
    0: "의문유발-부호", 1: "의문유발-은닉", 2: "선정표현",
    3: "속어/줄임말", 4: "사실과대", 5: "주어왜곡"
}
df_multi = df[df['type_label'] != -1]
print(f"  다중 분류 대상: {len(df_multi):,}건")
multi_dist = df_multi['type_label'].value_counts().sort_index()
for label, count in multi_dist.items():
    name = TYPE_NAMES.get(label, "?")
    print(f"  {label} ({name}): {count:,}건 ({count/len(df_multi)*100:.1f}%)")
print(f"\n  최대/최소 비율: {multi_dist.max()/multi_dist.min():.1f}배 불균형")

## Step 3. 제목/본문 길이 확인 (토크나이저 설정 전 참고)

BERT는 `max_length=512` 토큰 제한이 있습니다.
- 제목은 짧으므로 잘릴 일이 거의 없음
- 본문은 50%가 1000자 초과 → `truncation='only_second'`로 본문만 자름

여기서는 **글자 수** 기준으로 분포를 빠르게 확인합니다.
(실제 토큰 수는 토크나이저마다 다르므로 Step 5에서 별도 확인)

In [ ]:
title_lens = df['title_clean'].str.len()
content_lens = df['content_clean'].str.len()

print("제목 (title_clean) 글자 수 통계")
print(f"  평균: {title_lens.mean():.1f}자 | 중앙값: {title_lens.median():.0f}자 | 최대: {title_lens.max()}자")
print(f"  95%ile: {title_lens.quantile(0.95):.0f}자 | 99%ile: {title_lens.quantile(0.99):.0f}자")

print(f"\n본문 (content_clean) 글자 수 통계")
print(f"  평균: {content_lens.mean():.1f}자 | 중앙값: {content_lens.median():.0f}자 | 최대: {content_lens.max()}자")
print(f"  95%ile: {content_lens.quantile(0.95):.0f}자 | 99%ile: {content_lens.quantile(0.99):.0f}자")
print(f"  1000자 초과: {(content_lens > 1000).mean()*100:.1f}%")
print(f"  1500자 초과: {(content_lens > 1500).mean()*100:.1f}%")

print(f"\n샘플 데이터 (첫 2건):")
for i in range(2):
    print(f"\n--- 샘플 {i+1} (binary_label={df.iloc[i]['binary_label']}, type_label={df.iloc[i]['type_label']}) ---")
    print(f"  제목: {df.iloc[i]['title_clean'][:80]}...")
    print(f"  본문: {df.iloc[i]['content_clean'][:120]}...")

## Step 4. 토크나이저 로드

3개 모델의 토크나이저를 각각 로드합니다. 최초 실행 시 HuggingFace에서 모델 파일을 다운로드합니다.

| 모델 | HuggingFace ID | 토크나이저 종류 |
|---|---|---|
| KoBERT | `skt/kobert-base-v1` | SentencePiece |
| KLUE-RoBERTa | `klue/roberta-base` | BPE (Byte-Pair Encoding) |
| KoELECTRA | `monologg/koelectra-base-v3-discriminator` | WordPiece |

In [ ]:
from transformers import AutoTokenizer

MODEL_NAMES = {
    'kobert':    'skt/kobert-base-v1',
    'klue':      'klue/roberta-base',
    'koelectra': 'monologg/koelectra-base-v3-discriminator',
}

tokenizers = {}
for key, model_id in MODEL_NAMES.items():
    print(f"[{key}] {model_id} 로딩 중...")
    tokenizers[key] = AutoTokenizer.from_pretrained(model_id)
    print(f"  → vocab size: {tokenizers[key].vocab_size:,}")

print("\n모든 토크나이저 로드 완료!")

## Step 5. 토크나이징 테스트 (샘플 1건)

실제 데이터 1건으로 3개 토크나이저의 출력을 비교합니다.

핵심 설정:
- `text` = 제목 (`title_clean`), `text_pair` = 본문 (`content_clean`)
- `truncation='only_second'`: **제목은 절대 자르지 않고**, 본문만 512 토큰에 맞춰 자름
- `max_length=512`, `padding='max_length'`
- 결과: `input_ids`, `attention_mask`, `token_type_ids`

In [ ]:
sample_title = df.iloc[0]['title_clean']
sample_content = df.iloc[0]['content_clean']

print(f"샘플 제목: {sample_title[:80]}")
print(f"샘플 본문: {sample_content[:120]}...\n")

for key, tok in tokenizers.items():
    encoded = tok(
        text=sample_title,
        text_pair=sample_content,
        truncation='only_second',
        max_length=512,
        padding='max_length',
        return_tensors='pt',
    )
    
    input_ids = encoded['input_ids'][0]
    attn_mask = encoded['attention_mask'][0]
    non_pad = attn_mask.sum().item()
    
    title_tokens = tok.tokenize(sample_title)
    
    print(f"{'=' * 60}")
    print(f"[{key}] {MODEL_NAMES[key]}")
    print(f"  제목 토큰 수: {len(title_tokens)}")
    print(f"  제목 토큰: {title_tokens[:15]}{'...' if len(title_tokens) > 15 else ''}")
    print(f"  전체 토큰 수 (패딩 제외): {non_pad}")
    print(f"  input_ids shape: {input_ids.shape}")
    print(f"  attention_mask shape: {attn_mask.shape}")
    if 'token_type_ids' in encoded:
        print(f"  token_type_ids shape: {encoded['token_type_ids'][0].shape}")
    else:
        print(f"  token_type_ids: 없음 (RoBERTa 계열은 미사용)")

## Step 5-1. 제목 토큰 수 분포 확인

`truncation='only_second'`를 쓰려면 **제목 토큰 수가 510 이하**여야 합니다.
(512 - [CLS] 1개 - [SEP] 1개 = 510)

만약 제목이 510 토큰을 초과하는 케이스가 있다면 `only_second` 전략이 실패할 수 있으므로 미리 확인합니다.
전체 29만 건을 돌리면 오래 걸리므로, 랜덤 5,000건으로 샘플 확인합니다.

In [ ]:
import numpy as np

sample_df = df.sample(n=5000, random_state=42)

tok_check = tokenizers['klue']
title_token_lens = sample_df['title_clean'].apply(lambda x: len(tok_check.tokenize(x)))

print("제목 토큰 수 분포 (KLUE-RoBERTa 기준, 5,000건 샘플)")
print(f"  평균: {title_token_lens.mean():.1f} | 최대: {title_token_lens.max()}")
print(f"  95%ile: {title_token_lens.quantile(0.95):.0f} | 99%ile: {title_token_lens.quantile(0.99):.0f}")
print(f"  50 토큰 초과: {(title_token_lens > 50).sum()}건")
print(f"  100 토큰 초과: {(title_token_lens > 100).sum()}건")
print(f"  510 토큰 초과: {(title_token_lens > 510).sum()}건")

if title_token_lens.max() <= 510:
    print("\n✅ 제목 토큰 수 최대값이 510 이하 → truncation='only_second' 안전하게 사용 가능")
else:
    print(f"\n⚠️ 제목이 510 토큰을 초과하는 케이스 존재 → 별도 처리 필요")

## Step 6. PyTorch Dataset 클래스

3개 모델 공통으로 사용할 수 있는 Dataset 클래스를 정의합니다.

**핵심 설계**:
- `__getitem__`에서 on-the-fly 토크나이징 (메모리 절약)
- `truncation='only_second'` → 제목 보존, 본문만 자름
- `token_type_ids` 미반환 모델(KoBERT, KLUE-RoBERTa)은 자동으로 0 텐서 생성
- `task` 파라미터로 이진/다중 분류 전환:
  - `'binary'`: 전체 데이터, `binary_label` 사용
  - `'multi'`: `type_label != -1`인 Clickbait_Direct만, `type_label` 사용

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class ClickbaitDataset(Dataset):
    """낚시성 기사 탐지용 BERT Dataset (이진/다중 분류 공용)"""
    
    def __init__(self, dataframe, tokenizer, max_length=512, task='binary'):
        """
        Args:
            dataframe: title_clean, content_clean, binary_label, type_label 컬럼 필요
            tokenizer: HuggingFace 토크나이저
            max_length: 최대 토큰 길이 (기본 512)
            task: 'binary' (이진) 또는 'multi' (다중 분류)
        """
        if task == 'multi':
            dataframe = dataframe[dataframe['type_label'] != -1].reset_index(drop=True)
        
        self.titles = dataframe['title_clean'].tolist()
        self.contents = dataframe['content_clean'].tolist()
        self.labels = dataframe['binary_label' if task == 'binary' else 'type_label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.task = task
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        encoded = self.tokenizer(
            text=self.titles[idx],
            text_pair=self.contents[idx],
            truncation='only_second',
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        
        item = {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
        }
        
        if 'token_type_ids' in encoded:
            item['token_type_ids'] = encoded['token_type_ids'].squeeze(0)
        else:
            item['token_type_ids'] = torch.zeros_like(item['input_ids'])
        
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("ClickbaitDataset 클래스 정의 완료")

ClickbaitDataset 클래스 정의 완료


## Step 6-1. Dataset 동작 테스트

소규모 데이터(100건)로 Dataset이 제대로 작동하는지 확인합니다.
- 이진 분류 / 다중 분류 각각 테스트
- 3개 토크나이저 각각 출력 shape 확인

In [10]:
df_small = df.head(100)

print("=" * 60)
print("이진 분류 Dataset 테스트 (100건)")
print("=" * 60)
for key, tok in tokenizers.items():
    ds = ClickbaitDataset(df_small, tok, task='binary')
    sample = ds[0]
    print(f"\n[{key}] 데이터셋 크기: {len(ds)}")
    print(f"  input_ids:      {sample['input_ids'].shape}")
    print(f"  attention_mask:  {sample['attention_mask'].shape}")
    print(f"  token_type_ids:  {sample['token_type_ids'].shape}")
    print(f"  label:           {sample['labels'].item()}")

print(f"\n{'=' * 60}")
print("다중 분류 Dataset 테스트 (100건 중 type_label != -1)")
print("=" * 60)
ds_multi = ClickbaitDataset(df_small, tokenizers['klue'], task='multi')
print(f"  다중 분류 데이터셋 크기: {len(ds_multi)}")
if len(ds_multi) > 0:
    sample_m = ds_multi[0]
    print(f"  label: {sample_m['labels'].item()} (유형: {TYPE_NAMES.get(sample_m['labels'].item(), '?')})")
else:
    print("  (상위 100건에 Clickbait_Direct가 없음 — 전체 데이터로 테스트)")

이진 분류 Dataset 테스트 (100건)

[kobert] 데이터셋 크기: 100
  input_ids:      torch.Size([512])
  attention_mask:  torch.Size([512])
  token_type_ids:  torch.Size([512])
  label:           1

[klue] 데이터셋 크기: 100
  input_ids:      torch.Size([512])
  attention_mask:  torch.Size([512])
  token_type_ids:  torch.Size([512])
  label:           1

[koelectra] 데이터셋 크기: 100
  input_ids:      torch.Size([512])
  attention_mask:  torch.Size([512])
  token_type_ids:  torch.Size([512])
  label:           1

다중 분류 Dataset 테스트 (100건 중 type_label != -1)
  다중 분류 데이터셋 크기: 0
  (상위 100건에 Clickbait_Direct가 없음 — 전체 데이터로 테스트)


## Step 7. DataLoader 구성 및 테스트

실제 학습에서는 5-Fold Stratified CV로 train/val을 나누어 DataLoader를 만들지만,
여기서는 **DataLoader가 정상 동작하는지 확인**하는 것이 목적입니다.

- `batch_size=16` (Colab Pro V100 기준, 실제 학습 시 조정 가능)
- `shuffle=True` (train), `shuffle=False` (val)
- K-Fold 분할은 Colab 학습 코드에서 구현 예정

In [ ]:
BATCH_SIZE = 16

ds_test = ClickbaitDataset(df.head(100), tokenizers['klue'], task='binary')
loader_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=True)

batch = next(iter(loader_test))

print("DataLoader 배치 테스트 (KLUE-RoBERTa, batch_size=16)")
print(f"  input_ids:      {batch['input_ids'].shape}")
print(f"  attention_mask:  {batch['attention_mask'].shape}")
print(f"  token_type_ids:  {batch['token_type_ids'].shape}")
print(f"  labels:          {batch['labels'].shape}")
print(f"  labels 값:       {batch['labels'].tolist()}")
print(f"\n✅ DataLoader 정상 동작 확인")

DataLoader 배치 테스트 (KLUE-RoBERTa, batch_size=16)
  input_ids:      torch.Size([16, 512])
  attention_mask:  torch.Size([16, 512])
  token_type_ids:  torch.Size([16, 512])
  labels:          torch.Size([16])
  labels 값:       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

✅ DataLoader 정상 동작 확인


## Step 8. 전체 파이프라인 함수 정리

Colab에서 바로 사용할 수 있도록 학습 파이프라인 헬퍼 함수를 정리합니다.

- `get_dataset()`: 모델 이름과 task를 지정하면 Dataset 생성
- `get_dataloaders()`: K-Fold의 train/val 인덱스를 받아 DataLoader 쌍 반환
- `compute_class_weights()`: 다중 분류용 class weight 계산

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Subset

def get_dataset(df, model_key, task='binary', max_length=512):
    """모델 이름과 task를 지정하면 Dataset 생성"""
    tok = AutoTokenizer.from_pretrained(MODEL_NAMES[model_key])
    return ClickbaitDataset(df, tok, max_length=max_length, task=task)

def get_fold_dataloaders(dataset, train_idx, val_idx, batch_size=16):
    """K-Fold의 train/val 인덱스를 받아 DataLoader 쌍 반환"""
    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=batch_size, shuffle=False
    )
    return train_loader, val_loader

def get_class_weights(labels, device='cpu'):
    """다중 분류용 class weight 계산 (class_weight='balanced'와 동일)"""
    classes = sorted(set(labels))
    weights = compute_class_weight('balanced', classes=np.array(classes), y=np.array(labels))
    return torch.tensor(weights, dtype=torch.float).to(device)

print("헬퍼 함수 정의 완료")

# K-Fold 설정 확인
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"\nStratifiedKFold 설정: n_splits=5, shuffle=True, random_state=42")

# 다중 분류 class weight 미리 확인
df_multi = df[df['type_label'] != -1]
cw = get_class_weights(df_multi['type_label'].tolist())
print(f"\n다중 분류 class weights:")
for i, w in enumerate(cw):
    print(f"  {i} ({TYPE_NAMES[i]}): {w:.4f}")

헬퍼 함수 정의 완료

StratifiedKFold 설정: n_splits=5, shuffle=True, random_state=42

다중 분류 class weights:
  0 (의문유발-부호): 0.4497
  1 (의문유발-은닉): 0.5720
  2 (선정표현): 1.8808
  3 (속어/줄임말): 1.8085
  4 (사실과대): 1.5810
  5 (주어왜곡): 3.2136


## 전처리 완료 요약

### 확인된 사항
| 항목 | 결과 |
|---|---|
| 데이터 | 291,466건 (work_pool 3개 병합, test_final 봉인) |
| 이진 분류 | 50:50 균형 → class_weight 불필요 |
| 다중 분류 | 40,106건, 7.1배 불균형 → class_weight='balanced' 적용 |
| 제목 최대 토큰 | 36 (510 한참 이하) → `truncation='only_second'` 안전 |
| 토크나이저 | KoBERT(8K), KLUE-RoBERTa(32K), KoELECTRA(35K) 모두 정상 |
| Dataset/DataLoader | 이진/다중 분류 모두 정상 동작 |

### Colab 학습 시 사용할 코드
```python
# 1. 모델별 Dataset 생성
dataset = get_dataset(df, model_key='klue', task='binary')

# 2. 5-Fold 반복
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_loader, val_loader = get_fold_dataloaders(dataset, train_idx, val_idx)
    # 학습 루프...

# 3. 다중 분류 시 class weight
class_weights = get_class_weights(labels, device='cuda')
criterion = nn.CrossEntropyLoss(weight=class_weights)
```

### 다음 단계 (Colab Pro)
- [ ] 3개 모델 x 이진/다중 분류 x 5-Fold 학습
- [ ] 성능 비교 (Accuracy, F1, Precision, Recall)
- [ ] 최고 모델 선택 → test_final로 최종 1회 평가